# Qwen3-Embedding-4B Embedding Stage

Goal: create two embedding inputs for the next RQ-VAE stage.

- `meta_text = title + year + genres`
- `description_text = overview`
- cells are not executed

In [1]:
from pathlib import Path
import json
import re
import time

import numpy as np
import pandas as pd
import torch

ROOT = Path(r"C:/Users/User/plum-ml1m-repro")

SOURCE_CSV = Path(r"D:/user/Downloads/ml1m_movie_overviews_sample720_r9_checked_v2_audited_next200_r7_final_audit100.csv")
MOVIES_REINDEXED_PATH = ROOT / "data/processed/movies_reindexed.parquet"
OUT_DIR = ROOT / "data/processed/item_features"

RUN_NAME = "qwen4b_audited_v1"
PROFILE_PATH = OUT_DIR / f"{RUN_NAME}_item_profiles.parquet"
EMBEDDING_PATH = OUT_DIR / f"{RUN_NAME}_meta_desc_embeddings.npz"
MANIFEST_PATH = OUT_DIR / f"{RUN_NAME}_embedding_manifest.json"

MODEL_NAME = "Qwen/Qwen3-Embedding-4B"
MAX_SEQ_LENGTH = 1024
TRUNCATE_DIM = None
META_BATCH_SIZE = 8
DESC_BATCH_SIZE = 4
NORMALIZE_EMBEDDINGS = True
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

OUT_DIR.mkdir(parents=True, exist_ok=True)
DEVICE


# cell 1 finished in 1.38s


'cuda'

## 1. Load data

Use project `item_idx`; use audited CSV fields for movie text.

In [2]:
audited = pd.read_csv(SOURCE_CSV)
movies_reindexed = pd.read_parquet(MOVIES_REINDEXED_PATH, columns=["movie_id", "item_idx"])

base_cols = ["movie_id", "title", "year", "genres", "overview", "source", "status"]
audited_core = audited[base_cols].copy()

profiles = (
    movies_reindexed[["movie_id", "item_idx"]]
    .merge(audited_core, on="movie_id", how="left", validate="one_to_one")
    .sort_values("item_idx")
    .reset_index(drop=True)
)

assert profiles["overview"].notna().all()
assert np.array_equal(profiles["item_idx"].to_numpy(), np.arange(len(profiles)))

profiles.head()


# cell 3 finished in 0.06s


   movie_id  item_idx  ...                                             source  status
0         1         0  ...  https://huggingface.co/datasets/mt0rm0/movie_d...   found
1         2         1  ...  https://huggingface.co/datasets/mt0rm0/movie_d...   found
2         3         2  ...  https://huggingface.co/datasets/mt0rm0/movie_d...   found
3         4         3  ...  https://huggingface.co/datasets/mt0rm0/movie_d...   found
4         5         4  ...  https://huggingface.co/datasets/mt0rm0/movie_d...   found

[5 rows x 8 columns]

## 2. Build text fields

Keep two separate texts: one for metadata, one for plot overview.

In [3]:
YEAR_RE = re.compile(r"\s*\((\d{4})\)\s*$")

def clean_title(x):
    return YEAR_RE.sub("", str(x).strip()).strip()

def clean_year(x):
    return "unknown" if pd.isna(x) else str(int(float(x)))

def clean_genres(x):
    return ", ".join(part.strip() for part in str(x).split("|") if part.strip())

def clean_text(x):
    return re.sub(r"\s+", " ", str(x).replace("\n", " ").replace("\r", " ")).strip()

profiles["title_clean"] = profiles["title"].map(clean_title)
profiles["year_text"] = profiles["year"].map(clean_year)
profiles["genres_text"] = profiles["genres"].map(clean_genres)
profiles["overview_clean"] = profiles["overview"].map(clean_text)

profiles["meta_text"] = profiles.apply(
    lambda r: f"Movie title: {r.title_clean}. Release year: {r.year_text}. Genres: {r.genres_text}.",
    axis=1,
)
profiles["description_text"] = profiles["overview_clean"].map(lambda x: f"Plot overview: {x}")

profiles.to_parquet(PROFILE_PATH, index=False)
profiles[["item_idx", "title", "meta_text", "description_text"]].head()


# cell 5 finished in 0.10s


   item_idx  ...                                   description_text
0         0  ...  Plot overview: Led by Woody, Andy's toys live ...
1         1  ...  Plot overview: When siblings Judy and Peter di...
2         2  ...  Plot overview: A family wedding reignites the ...
3         3  ...  Plot overview: Cheated on, mistreated and step...
4         4  ...  Plot overview: Just when George Banks has reco...

[5 rows x 4 columns]

## 3. Load Qwen3-Embedding-4B

If CUDA OOM happens, lower the two batch-size constants in the first cell.

In [4]:
from sentence_transformers import SentenceTransformer

torch.cuda.empty_cache()
model_kwargs = {"torch_dtype": torch.float16} if DEVICE == "cuda" else {}

model = SentenceTransformer(
    MODEL_NAME,
    device=DEVICE,
    model_kwargs=model_kwargs,
)
model.max_seq_length = MAX_SEQ_LENGTH
model

C:\Users\User\AppData\Local\Programs\Python\Python313\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\User\.cache\huggingface\hub\models--Qwen--Qwen3-Embedding-4B. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|##########| 398/398 [00:00<00:00, 662.36it/s]

# cell 7 finished in 51

SentenceTransformer(
  (0): Transformer({'transformer_task': 'feature-extraction', 'modality_config': {'text': {'method': 'forward', 'method_output_name': 'last_hidden_state'}}, 'module_output_name': 'token_embeddings', 'architecture': 'Qwen3Model'})
  (1): Pooling({'embedding_dimension': 2560, 'pooling_mode': 'lasttoken', 'include_prompt': True})
  (2): Normalize({})
)

## 4. Encode helper

No query prompt: these are item content vectors for RQ-VAE.

In [5]:
def encode_texts(texts, batch_size):
    kwargs = dict(
        batch_size=batch_size,
        normalize_embeddings=NORMALIZE_EMBEDDINGS,
        show_progress_bar=True,
        convert_to_numpy=True,
    )
    if TRUNCATE_DIM is not None:
        kwargs["truncate_dim"] = TRUNCATE_DIM
    return model.encode(list(texts), **kwargs).astype("float32", copy=False)


# cell 9 finished in 0.00s


## 5. Encode meta

In [6]:
meta = encode_texts(profiles["meta_text"], META_BATCH_SIZE)
meta.shape

Batches: 100%|##########| 464/464 [00:19<00:00, 24.37it/s]

# cell 11 finished in 19.08s


(3706, 2560)

## 6. Encode descriptions

In [7]:
description = encode_texts(profiles["description_text"], DESC_BATCH_SIZE)
description.shape

Batches: 100%|##########| 927/927 [00:47<00:00, 19.60it/s]

# cell 13 finished in 47.35s


(3706, 2560)

## 7. Small sanity check

In [8]:
assert meta.shape == description.shape
assert meta.shape[0] == len(profiles)
assert np.isfinite(meta).all()
assert np.isfinite(description).all()

{
    "items": len(profiles),
    "embedding_dim": meta.shape[1],
    "meta_norm_mean": float(np.linalg.norm(meta, axis=1).mean()),
    "description_norm_mean": float(np.linalg.norm(description, axis=1).mean()),
}


# cell 15 finished in 0.02s


{'items': 3706, 'embedding_dim': 2560, 'meta_norm_mean': 1.0001639127731323, 'description_norm_mean': 1.0001496076583862}

## 8. Save bundle

Saved arrays: `item_idx`, `movie_id`, `meta`, `description`.

In [9]:
np.savez_compressed(
    EMBEDDING_PATH,
    item_idx=profiles["item_idx"].to_numpy("int64"),
    movie_id=profiles["movie_id"].to_numpy("int64"),
    meta=meta,
    description=description,
)

manifest = {
    "run_name": RUN_NAME,
    "created_at": time.strftime("%Y-%m-%d %H:%M:%S"),
    "source_csv": str(SOURCE_CSV),
    "profile_path": str(PROFILE_PATH),
    "embedding_path": str(EMBEDDING_PATH),
    "model_name": MODEL_NAME,
    "max_seq_length": MAX_SEQ_LENGTH,
    "truncate_dim": TRUNCATE_DIM,
    "normalize_embeddings": NORMALIZE_EMBEDDINGS,
    "n_items": int(len(profiles)),
    "embedding_dim": int(meta.shape[1]),
}

MANIFEST_PATH.write_text(json.dumps(manifest, ensure_ascii=False, indent=2), encoding="utf-8")
EMBEDDING_PATH, MANIFEST_PATH


# cell 17 finished in 10.40s


(WindowsPath('C:/Users/User/plum-ml1m-repro/data/processed/item_features/qwen4b_audited_v1_meta_desc_embeddings.npz'), WindowsPath('C:/Users/User/plum-ml1m-repro/data/processed/item_features/qwen4b_audited_v1_embedding_manifest.json'))

## 9. Reload check

In [10]:
bundle = np.load(EMBEDDING_PATH)
{k: bundle[k].shape for k in bundle.files}


# cell 19 finished in 0.25s


{'item_idx': (3706,), 'movie_id': (3706,), 'meta': (3706, 2560), 'description': (3706, 2560)}

## 10. Tiny semantic check

Same-movie meta/description cosine should usually be higher than random pairs.

In [11]:
same = np.sum(meta * description, axis=1)
rng = np.random.default_rng(42)
random = np.sum(meta * description[rng.permutation(len(description))], axis=1)

{
    "same_meta_desc_cosine_mean": float(same.mean()),
    "random_meta_desc_cosine_mean": float(random.mean()),
    "gap": float(same.mean() - random.mean()),
}


# cell 21 finished in 0.02s


{'same_meta_desc_cosine_mean': 0.3663513660430908, 'random_meta_desc_cosine_mean': 0.2277015745639801, 'gap': 0.13864979147911072}